# Aggregated Lab 2 Rhythm Analysis

This notebook mirrors the Lab 2 rhythm ideas, but aggregates across the sampled `classical`, `pop`, `game_looping`, and `game_not_looping` datasets.

In [ ]:
import os
from collections import Counter

import mido
import numpy as np
import pandas as pd
import pretty_midi
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

BASE_DIR = os.path.abspath("..")
GENRE_FOLDERS = [
    ("classical", os.path.join(BASE_DIR, "sampled_datasets", "classical_midi")),
    ("pop", os.path.join(BASE_DIR, "sampled_datasets", "pop_midi")),
    ("game_looping", os.path.join(BASE_DIR, "sampled_datasets", "game_looping_midi")),
    ("game_not_looping", os.path.join(BASE_DIR, "sampled_datasets", "game_not_looping_midi")),
]
GENRE_ORDER = [genre for genre, _ in GENRE_FOLDERS]


In [ ]:
def midi_note_on_beats(path):
    midi = mido.MidiFile(path)
    current_ticks = 0
    onset_beats = []

    for msg in mido.merge_tracks(midi.tracks):
        current_ticks += msg.time

        if msg.type == "note_on" and msg.velocity > 0:
            onset_beats.append(current_ticks / midi.ticks_per_beat)

    return onset_beats


def beats_per_bar(path):
    pm = pretty_midi.PrettyMIDI(path)

    if len(pm.time_signature_changes) == 0:
        return 4

    counts = Counter(ts.numerator for ts in pm.time_signature_changes)
    return counts.most_common(1)[0][0]


def onset_positions_within_bar(onset_beats, meter):
    positions = []

    for onset in onset_beats:
        beat_in_bar = int(np.floor(onset % meter)) + 1
        positions.append(beat_in_bar)

    return positions


In [ ]:
ioi_rows = []
onset_rows = []
meter_rows = []

for genre, folder in GENRE_FOLDERS:
    for file in os.listdir(folder):
        if not file.lower().endswith((".mid", ".midi")):
            continue

        path = os.path.join(folder, file)

        try:
            onset_beats = midi_note_on_beats(path)
            meter = beats_per_bar(path)

            meter_rows.append({
                "genre": genre,
                "file": file,
                "meter": meter,
                "non_4": meter != 4,
            })

            if len(onset_beats) >= 2:
                iois = np.diff(onset_beats)

                for ioi in iois:
                    ioi_rows.append({
                        "genre": genre,
                        "file": file,
                        "ioi_beats": float(ioi),
                        "positive_ioi": float(ioi) > 0,
                    })

            for beat_position in onset_positions_within_bar(onset_beats, meter):
                onset_rows.append({
                    "genre": genre,
                    "file": file,
                    "meter": meter,
                    "beat_position": beat_position,
                })

        except Exception as e:
            print("failed:", file, e)

ioi_df = pd.DataFrame(ioi_rows)
onset_df = pd.DataFrame(onset_rows)
meter_df = pd.DataFrame(meter_rows)

print(ioi_df.head())
print(onset_df.head())
print(meter_df.head())


## IOI Histogram

In [ ]:
plt.figure(figsize=(12, 6))

sns.histplot(
    data=ioi_df,
    x="ioi_beats",
    hue="genre",
    hue_order=GENRE_ORDER,
    bins=60,
    element="step",
    stat="count",
    common_norm=False
)

plt.xlim(0, min(8, ioi_df["ioi_beats"].quantile(0.99)))
plt.xlabel("Inter-onset interval (beats)")
plt.ylabel("Count")
plt.title("Aggregated IOI Distribution by Genre")
plt.tight_layout()
plt.savefig("aggregated_ioi_histogram.png", dpi=300)
plt.show()


## Positive IOIs Only

In [ ]:
positive_ioi_df = ioi_df[ioi_df["ioi_beats"] > 0].copy()

plt.figure(figsize=(12, 6))

sns.histplot(
    data=positive_ioi_df,
    x="ioi_beats",
    hue="genre",
    hue_order=GENRE_ORDER,
    bins=60,
    element="step",
    stat="count",
    common_norm=False
)

plt.xlim(0, min(8, positive_ioi_df["ioi_beats"].quantile(0.99)))
plt.xlabel("Positive inter-onset interval (beats)")
plt.ylabel("Count")
plt.title("Positive IOI Distribution by Genre")
plt.tight_layout()
plt.savefig("aggregated_positive_ioi_histogram.png", dpi=300)
plt.show()

print(
    positive_ioi_df.groupby(["genre", "ioi_beats"]).size().reset_index(name="count").sort_values(["genre", "count"], ascending=[True, False]).head(40)
)


## Meter Breakdown

In [ ]:
meter_summary = meter_df.groupby("genre").agg(
    total_files=("file", "count"),
    non_4_files=("non_4", "sum")
).reset_index()

meter_breakdown = meter_df.groupby(["genre", "meter"]).size().reset_index(name="count")

print(meter_summary)
print(meter_breakdown)

plt.figure(figsize=(12, 6))

sns.barplot(
    data=meter_breakdown,
    x="meter",
    y="count",
    hue="genre",
    hue_order=GENRE_ORDER
)

plt.xlabel("Beats per bar")
plt.ylabel("Number of files")
plt.title("Meter Breakdown by Genre")
plt.tight_layout()
plt.savefig("aggregated_meter_breakdown.png", dpi=300)
plt.show()


## Onset Positions Within Each Piece's Meter

In [ ]:
onset_profile = onset_df.groupby(["genre", "beat_position"]).size().reset_index(name="count")
onset_profile["beat_position"] = onset_profile["beat_position"].astype(int)
onset_profile = onset_profile.sort_values(["beat_position", "genre"])

plt.figure(figsize=(12, 6))

sns.barplot(
    data=onset_profile,
    x="beat_position",
    y="count",
    hue="genre",
    hue_order=GENRE_ORDER
)

plt.xlabel("Beat number within measure")
plt.ylabel("Number of note onsets")
plt.title("Aggregated Onset Positions Within Piece Meter")
plt.tight_layout()
plt.savefig("aggregated_onset_positions.png", dpi=300)
plt.show()

print(onset_profile)
